# Прототип «Помощник подбора: телевизоры» — запуск в Colab

**Перед запуском:** меню *Среда выполнения → Сменить среду выполнения → T4 GPU* (бесплатно). Без видеокарты прототип тоже работает, но с моделью поменьше и медленнее.

**Порядок:** впишите адрес своего репозитория в ячейку 1 и нажмите *Среда выполнения → Выполнить все*. В конце появится ссылка `https://….trycloudflare.com` — её можно открыть с любого компьютера или телефона.

⚠️ Colab — облако Google. Здесь можно показывать только демо с вымышленным каталогом. Реальные данные компании — только на нашем сервере.

Ссылка работает, пока открыта эта вкладка Colab. При новом запуске ссылка меняется.

In [ ]:
# 1. Настройки
REPO_URL = "https://github.com/ВАШ_ЛОГИН/tv-assistant-demo.git"  # ← замените на свой репозиторий
LLM_MODEL = ""          # пусто = автоматически: Qwen2.5-1.5B на видеокарте, Qwen2.5-0.5B на процессоре
ASR_BACKEND = "gigaam"  # "gigaam" (GigaAM-v3) | "whisper" (запасной) | "none" (без голоса)
PARSER = "llm"          # "llm" — модель + правила | "rules" — только правила

In [ ]:
# 2. Код и зависимости (первый раз 3–5 минут, повторно — секунды)
import os
os.chdir("/content")
if not os.path.exists("tv-assistant-demo"):
    !git clone -q $REPO_URL tv-assistant-demo
os.chdir("/content/tv-assistant-demo")
!git pull -q
!pip install -q -r requirements.txt
import importlib.util
if ASR_BACKEND == "gigaam" and importlib.util.find_spec("gigaam") is None:
    print("Ставлю GigaAM: 2–4 минуты, ячейку не прерывайте…")
    !pip install -q "git+https://github.com/salute-developers/GigaAM.git"
else:
    print("GigaAM уже установлен или не нужен — пропускаю")
!which ffmpeg > /dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg)
import torch
print("Видеокарта:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет — будет работать на процессоре")

In [ ]:
# 3. Скачать модели заранее, чтобы показ не ждал загрузки
os.environ["ASR_BACKEND"] = ASR_BACKEND
os.environ["PARSER"] = PARSER
if LLM_MODEL:
    os.environ["LLM_MODEL"] = LLM_MODEL
!python -m core.prefetch

In [ ]:
# 3б. Проверка перед показом: тесты логики и эталон фраз (ожидается «57 passed»)
!python -m pytest -q


In [ ]:
# 4. Запуск Streamlit и публичной HTTPS-ссылки (микрофон в браузере работает только по HTTPS)
import re, subprocess, time
subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
app = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    env=dict(os.environ), stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
if url:
    print("Ссылка для показа:", url)
    print("Откройте её сами один раз заранее: при первом открытии модели загружаются в память (около минуты).")
else:
    print("Ссылка не появилась — посмотрите /content/tunnel.log и запустите эту ячейку ещё раз.")

In [ ]:
# 5. Если что-то не так — последние строки журнала приложения
!tail -40 /content/streamlit.log

**Если не работает микрофон:** ссылка должна начинаться с `https://`; в браузере разрешите доступ к микрофону (значок замка в адресной строке). Если микрофон недоступен, выжимку можно ввести текстом — разбор тот же.

**Если GigaAM не скачался** (сервер моделей Сбера недоступен из Colab), приложение само переключится на ту же модель с Hugging Face, а затем на Whisper. Что именно работает, видно в боковой панели приложения.